# NeXo v3.0 · Notebook 01 — Preprocessing + Feature Engineering + Data Warehouse (CRISP-DM Phase 3)

> **CRISP-DM Phase 3 — Data Preparation.** Transforms raw TT_data/ CSVs into curated `warehouse.parquet` consumed by training notebooks 02 / 03 / 04.

## Contract

- **Input**: raw BSS + OSS CSVs from `TT_data/` (uploaded to MinIO `raw/` bucket).
- **Output**: `curated/warehouse.parquet` (subscriber + area-level features + CEM score target), `curated/cells.parquet` (per-cell for VAE), `curated/splits.json`.
- **Consumed by**: notebooks 02 (CEM), 03 (VAE), 04 (RAT) via NPZ exports.
- **Idempotent**: re-running with same SEED + same source data produces byte-identical parquets.

## Pipeline

```
TT_data/ CSVs
  → upload to MinIO raw/
  → read back from raw/
  → normalize OSS (parse %, derive area, 3GPP-formula latency/loss/jitter)
  → clean BSS (domain rules + clipping)
  → IterativeImputer (MICE-style) on numeric NaNs
  → Winsorize p99 on traffic columns (bounds saved)
  → derive features (data_intensity, rat-share, attach_gap, is_4g_capable, usim_bottleneck)
  → join BSS × OSS aggregates by area
  → compute CEM score target (0.4·attach + 0.3·4g_share + 0.2·integrity + 0.1·cdr_inv)
  → train/val/test split (random stratified by area + temporal holdout)
  → write curated/ to MinIO
```

## What this notebook does NOT do

- Train models — that's notebooks 02 / 03 / 04.
- Tune hyperparameters of the imputer / Winsorize threshold — that's a follow-up sensitivity-analysis pass.
- Validate against the Granger gate — that's notebook 10 (Tier 1 offline gate).


## 0 · Reproducibility, Constants & CEM Weights

**What this shows:** All hyperparameters in ONE place — SEED, CEM-score formula weights, Winsorize percentile, split fractions, imputer iteration cap. Papermill-overridable.

**What to look for:** The four `CEM_W_*` constants define the CEM score formula. These are the most important hyperparameters in the entire project. They MUST sum to 1.0 (asserted). Any change requires owner approval + sensitivity analysis.

In [ ]:
# --- Papermill parameters cell (tag: parameters) ---
# Default constants. Override at retrain time via papermill -p flags.
# All magic numbers concentrated here for review.

SEED                       = 42        # numpy / random / sklearn / torch base seed

# --- BSS cleaning constants ---
ATTACH_SR_MIN              = 0.0       # success rate min bound
ATTACH_SR_MAX              = 1.0       # success rate max bound (clip to [0, 1])
WINSORIZE_PCT              = 99        # cap heavy-tailed traffic at p99 (justification: §8 ablation)

# --- CEM score weighting (cell 13) ---
# CEM score = 0.40 * attach_success + 0.30 * traffic_4g_share + 0.20 * data_integrity + 0.10 * cdr_inverse
# Sources for weights: domain expert (Huawei SmartCare scoring guidance, OSS+BSS convergence doc)
# These weights are the SINGLE most important hyperparameter in the project.
# Owner must approve any change. Sensitivity analysis recommended (see explainer).
CEM_W_ATTACH               = 0.40      # subscriber network-attach success rate
CEM_W_4G_SHARE             = 0.30      # share of traffic on 4G (modern RAT preference)
CEM_W_INTEGRITY            = 0.20      # OSS data-integrity area average
CEM_W_CDR_INV              = 0.10      # 1 - call-drop-rate area average

# --- Split policy ---
SPLIT_TRAIN_FRAC           = 0.70      # train share
SPLIT_VAL_FRAC             = 0.15      # validation share
SPLIT_TEST_FRAC            = 0.15      # test share (random) + last-10%-months temporal holdout
SPLIT_STRATIFY_COL         = 'area'    # ensures geographic representation
TEMPORAL_HOLDOUT_MONTHS_N  = 1         # how many trailing months reserved for temporal test

# --- Imputer ---
IMPUTER_MAX_ITER           = 10        # IterativeImputer convergence cap

# Hardcoded seed setup
import os, random
import numpy as np
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Assert weight sum to 1.0 (defense-grade contract)
assert abs(CEM_W_ATTACH + CEM_W_4G_SHARE + CEM_W_INTEGRITY + CEM_W_CDR_INV - 1.0) < 1e-9, \
    'CEM weights must sum to 1.0 (currently: {})'.format(CEM_W_ATTACH + CEM_W_4G_SHARE + CEM_W_INTEGRITY + CEM_W_CDR_INV)

print(f'SEED                       = {SEED}')
print(f'CEM weights (attach/4g/integrity/cdr_inv) = ({CEM_W_ATTACH}, {CEM_W_4G_SHARE}, {CEM_W_INTEGRITY}, {CEM_W_CDR_INV})')
print(f'CEM weight sum             = {CEM_W_ATTACH+CEM_W_4G_SHARE+CEM_W_INTEGRITY+CEM_W_CDR_INV}')
print(f'Winsorize percentile       = {WINSORIZE_PCT}')
print(f'Split (train/val/test)     = ({SPLIT_TRAIN_FRAC}, {SPLIT_VAL_FRAC}, {SPLIT_TEST_FRAC})'); print(f'Temporal holdout months    = {TEMPORAL_HOLDOUT_MONTHS_N}')

## 1 · Imports + MinIO client

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from IPython.display import display

import io, json, os, warnings
from pathlib import Path
import boto3, joblib, numpy as np, pandas as pd
from botocore.client import Config
from botocore.exceptions import ClientError
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

S3_ENDPOINT = os.environ.get('S3_ENDPOINT', 'http://localhost:9000')
s3 = boto3.client('s3', endpoint_url=S3_ENDPOINT,
                  aws_access_key_id=os.environ.get('S3_ACCESS_KEY','minio'),
                  aws_secret_access_key=os.environ.get('S3_SECRET_KEY','minio_pw'),
                  config=Config(signature_version='s3v4'))
RAW='raw'; PROC='processed'; CUR='curated'
print(f'MinIO endpoint = {S3_ENDPOINT}')

MinIO endpoint = http://localhost:9000


## 2 · Bootstrap MinIO buckets + PURGE raw of stale objects

Creates raw/processed/curated if missing. Then enforces invariant: `raw/` contains **only**
`bss/*` + `oss/*` keys. Anything else (old `run-*.json`, leftover synthetic) is deleted.

In [3]:
from time import sleep

def ensure(b):
    try: s3.head_bucket(Bucket=b); print(f'  exists: {b}')
    except ClientError: s3.create_bucket(Bucket=b); print(f'  CREATED: {b}')
for b in [RAW, PROC, CUR]:
    for attempt in range(5):
        try:
            ensure(b)
            break
        except Exception as e:
            if attempt == 4:
                raise
            print(f'  retrying {b} after S3 error: {type(e).__name__}')
            sleep(2)

# Purge raw of non-dataset keys
paginator = s3.get_paginator('list_objects_v2')
to_delete = []
kept = 0
for page in paginator.paginate(Bucket=RAW):
    for obj in page.get('Contents', []):
        k = obj['Key']
        if k.startswith(('bss/', 'oss/')): kept += 1
        else: to_delete.append({'Key': k})
if to_delete:
    for i in range(0, len(to_delete), 1000):
        s3.delete_objects(Bucket=RAW, Delete={'Objects': to_delete[i:i+1000], 'Quiet': True})
    print(f'  purged {len(to_delete)} stale objects ({kept} datasets retained)')
else:
    print(f'  raw/ already clean ({kept} dataset objects)')

  retrying raw after S3 error: EndpointConnectionError
  retrying raw after S3 error: EndpointConnectionError
  retrying raw after S3 error: EndpointConnectionError
  retrying raw after S3 error: EndpointConnectionError


EndpointConnectionError: Could not connect to the endpoint URL: "http://localhost:9000/raw"

## 3 · Upload BSS (16 files) + generated OSS (48 files) → raw/

- BSS: `TT_data/BSS/smartcare_cem_*.csv` → `raw/bss/`
- OSS: `TT_data/OSS/generated/oss_<rat>_<token>.csv` → `raw/oss/`

Idempotent — files with matching size are skipped.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
TT_DATA = ROOT/'TT_data'

def upload(p, key):
    try:
        if s3.head_object(Bucket=RAW, Key=key)['ContentLength']==p.stat().st_size: return False
    except ClientError: pass
    s3.upload_file(str(p), RAW, key); return True

up=sk=0
for f in sorted((TT_DATA/'BSS').glob('smartcare_cem_*.csv')):
    if upload(f, f'bss/{f.name}'): up+=1; print(f'  ↑ raw/bss/{f.name}')
    else: sk+=1
for f in sorted((TT_DATA/'OSS'/'generated').glob('oss_*.csv')):
    if upload(f, f'oss/{f.name}'): up+=1; print(f'  ↑ raw/oss/{f.name}')
    else: sk+=1
print(f'\n{up} uploaded, {sk} skipped (already present).')

## 4 · Read raw BSS back from MinIO

All 16 BSS monthly CSVs stitched. Tags: `month_year` (e.g. `2026-03`), `source_origin`.

In [ ]:
import re
MONTH_MAP = {'jan':'2026-01','feb':'2026-02','mars':'2026-03','avr':'2026-04','mai':'2026-05',
             'jun':'2026-06','jul':'2026-07','aug':'2026-08','aou':'2026-08','sep':'2026-09'}
REAL = {'feb','mars'}
def list_keys(bucket, prefix):
    keys=[]
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket, Prefix=prefix):
        for o in page.get('Contents', []): keys.append(o['Key'])
    return keys
frames = []
for k in list_keys(RAW, 'bss/'):
    m = re.search(r'smartcare_cem_([a-z]+)', k)
    if not m or m.group(1) not in MONTH_MAP: continue
    tok = m.group(1)
    body = s3.get_object(Bucket=RAW, Key=k)['Body'].read()
    df = pd.read_csv(io.BytesIO(body))
    df['month_year'] = MONTH_MAP[tok]
    df['source_origin'] = 'real' if tok in REAL else 'simulated'
    frames.append(df)
    print(f'  ⇣ raw/{k:50s} rows={len(df):>7,}')
df_bss_raw = pd.concat(frames, ignore_index=True); del frames
print(f'\nBSS: {len(df_bss_raw):,} rows × {df_bss_raw.shape[1]} cols')

## 5 · Read raw OSS back from MinIO (per RAT)

48 generated CSVs (3 RATs × 16 months). Each has Huawei metadata header (`skiprows=6`).
Each file's columns differ per RAT — we tag `rat_type` from the filename.

In [ ]:
RAT_TOK_RE = re.compile(r'oss_(\w{2})_([a-z_]+)\.csv$')
oss_by_rat = {'2G': [], '3G': [], '4G': []}
for k in sorted(list_keys(RAW, 'oss/')):
    m = RAT_TOK_RE.search(k)
    if not m: continue
    rat = m.group(1).upper()
    tok = m.group(2)
    if rat not in oss_by_rat: continue
    body = s3.get_object(Bucket=RAW, Key=k)['Body'].read()
    df = pd.read_csv(io.BytesIO(body), skiprows=6, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()
    df['rat_type'] = rat
    df['source_file'] = k.split('/')[-1]
    df['source_origin'] = 'real' if tok in REAL else 'simulated'
    oss_by_rat[rat].append(df)
    print(f'  ⇣ {k:50s} rat={rat} rows={len(df):>6,}')
for rat in oss_by_rat:
    oss_by_rat[rat] = pd.concat(oss_by_rat[rat], ignore_index=True) if oss_by_rat[rat] else pd.DataFrame()
    print(f'  {rat} total: {len(oss_by_rat[rat]):,} rows × {oss_by_rat[rat].shape[1] if not oss_by_rat[rat].empty else 0} cols')

## 6 · Normalize OSS — strip %, parse Time, derive area

Each RAT has different columns. Build unified DataFrame with common KPIs we need downstream.

In [ ]:
def _find(df, *frags):
    for c in df.columns:
        for f in frags:
            if f.lower() in c.lower(): return c
    return None

def _normalize(df, rat):
    if df.empty: return df
    out = pd.DataFrame()
    out['rat_type'] = df['rat_type']
    out['source_origin'] = df['source_origin']
    # Time
    time_c = _find(df, 'time')
    out['timestamp'] = pd.to_datetime(df[time_c], errors='coerce')
    out['month_year'] = out['timestamp'].dt.strftime('%Y-%m')
    # Cell + site
    out['cell_name'] = df[_find(df, 'cell name')].astype(str) if _find(df, 'cell name') else ''
    out['cell_id']   = df[_find(df, 'cell id', 'cell ci', 'localcell id')].astype(str) if _find(df, 'cell id','cell ci','localcell id') else ''
    site_c = _find(df, 'site name', 'nodeb name', 'enodeb name')
    out['site']      = df[site_c].astype(str) if site_c else ''
    # Area = first token of site name
    out['area'] = out['site'].str.split('_').str[0].str.replace(r'^(2G|3G|4G|cobts|cobtsko)', '', regex=True).str.strip()
    # Integrity (always string with %)
    integ_c = _find(df, 'integrity')
    if integ_c is not None:
        out['integrity'] = pd.to_numeric(df[integ_c].astype(str).str.rstrip('%').replace('', np.nan), errors='coerce')
    else:
        out['integrity'] = np.nan
    # CDR
    cdr_c = _find(df, 'call drop', 'opt_npm')
    out['call_drop_rate'] = pd.to_numeric(df[cdr_c], errors='coerce') if cdr_c else np.nan
    # Throughput (3G/4G only)
    tput_c = _find(df, 'throughput', 'thp')
    out['throughput_mbps'] = pd.to_numeric(df[tput_c], errors='coerce') if tput_c else np.nan
    if rat == '3G' and tput_c:
        out['throughput_mbps'] = out['throughput_mbps'] / 1000.0  # kbps → Mbps
    # RSRP (4G only)
    rsrp_c = _find(df, 'rsrp')
    out['rsrp_dbm'] = pd.to_numeric(df[rsrp_c], errors='coerce') if rsrp_c else np.nan
    # Users (4G only)
    ua_c = _find(df, 'user.avg')
    um_c = _find(df, 'user.max')
    out['active_users']     = pd.to_numeric(df[ua_c], errors='coerce') if ua_c else np.nan
    out['active_users_max'] = pd.to_numeric(df[um_c], errors='coerce') if um_c else np.nan
    # Anomaly flag
    out['anomaly_flag'] = ((out['integrity'] < 100) | (out['call_drop_rate'].fillna(0) > 2)).astype(int)
    return out

df_oss = pd.concat([_normalize(oss_by_rat[r], r) for r in ('2G','3G','4G')], ignore_index=True)
print(f'OSS normalized: {len(df_oss):,} rows × {df_oss.shape[1]} cols')
display(df_oss.head())
print('\nAnomaly rate per RAT:')
print(df_oss.groupby('rat_type').anomaly_flag.mean().round(4) * 100)

## 7 · Clean BSS — domain rules

- Drop NaN imsi/area
- Traffic/DOU/duration NaN → 0
- Attach success NaN → 1.0
- Categorical NaN → 'Unknown'

In [ ]:
df_bss = df_bss_raw.copy()
before = len(df_bss)
df_bss = df_bss.dropna(subset=['imsi','area'])
for c in ['dou_total','duration','voice_onlinetime_2g','voice_onlinetime_3g',
          'traffic_2g','traffic_3g','traffic_4g','traffic_5g']:
    if c in df_bss.columns:
        df_bss[c] = pd.to_numeric(df_bss[c], errors='coerce').fillna(0).clip(lower=0)
for c in ['s1_mme_sr','iu_attach_sr','gb_attach_sr']:
    if c in df_bss.columns:
        df_bss[c] = pd.to_numeric(df_bss[c], errors='coerce').clip(0,1).fillna(1.0)
for c in ['usertype','generation','tertype','brand','model']:
    if c in df_bss.columns:
        df_bss[c] = df_bss[c].fillna('Unknown')
print(f'Domain rules: {len(df_bss):,} rows (dropped {before-len(df_bss):,})')

## 8 · BSS IterativeImputer + Winsorize p99

MICE-style imputation for residual NaNs. Save imputer for inference reuse.

In [ ]:
num_cols = df_bss.select_dtypes(include='number').columns.tolist()
nan_cols = [c for c in num_cols if df_bss[c].isna().any()]
MODELS = Path('models'); MODELS.mkdir(exist_ok=True)
if nan_cols:
    imp = IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)
    df_bss[num_cols] = imp.fit_transform(df_bss[num_cols])
    joblib.dump(imp, MODELS/'bss_iterative_imputer.joblib')
    print(f'IterativeImputer fit on {len(num_cols)} cols ({len(nan_cols)} had NaN)')
wins = [c for c in ['dou_total','duration','traffic_2g','traffic_3g','traffic_4g','traffic_5g'] if c in df_bss.columns]
bounds = {}
for c in wins:
    p99 = df_bss[c].quantile(0.99)
    n = (df_bss[c]>p99).sum()
    df_bss[c] = df_bss[c].clip(upper=p99)
    bounds[c] = float(p99)
    print(f'  Winsor {c:18s} p99={p99:>10.2f} capped {n:>6,}')
joblib.dump(bounds, MODELS/'bss_winsor_bounds.joblib')

## 9 · BSS-derived features

`data_intensity`, traffic shares per RAT, `attach_gap`, `is_4g_capable`, `usim_bottleneck`.

In [ ]:
df_bss['data_intensity'] = df_bss['dou_total']/df_bss['duration'].clip(lower=1)
for r in ['2g','3g','4g','5g']:
    c=f'traffic_{r}'
    if c in df_bss.columns:
        df_bss[f'traffic_share_{r}'] = df_bss[c]/df_bss['dou_total'].clip(lower=1)
attach = [c for c in ['s1_mme_sr','iu_attach_sr','gb_attach_sr'] if c in df_bss.columns]
if attach: df_bss['attach_gap'] = 1.0 - df_bss[attach].mean(axis=1)
df_bss['is_4g_capable'] = (
    df_bss.get('generation','').astype(str).str.contains('4G|5G', regex=True)
    | (df_bss.get('traffic_4g', 0) > 0)
).astype(int)
df_bss['usim_bottleneck'] = ((df_bss['is_4g_capable']==1) & (df_bss.get('usim_flag','').astype(str)=='')).astype(int)
for c in ['data_intensity','traffic_share_4g','attach_gap','is_4g_capable','usim_bottleneck']:
    if c in df_bss.columns: print(f'  {c:25s} mean={df_bss[c].mean():.4f}')

## 10 · OSS-derived KPIs (latency/loss/jitter from 3GPP formulas)

Deterministic. Same formula as `vw_oss_cell_derived` SQL view.

In [ ]:
base = df_oss['rat_type'].map({'4G':18.0,'3G':55.0,'2G':95.0}).fillna(30.0)
integ = df_oss['integrity'].fillna(100.0)
cdr = df_oss['call_drop_rate'].fillna(0.0)
df_oss['latency_ms_derived'] = (base + 0.6*(100-integ).clip(lower=0) + 4.5*cdr).clip(lower=0)
df_oss['packet_loss_pct_derived'] = (0.5*cdr + 0.08*(100-integ).clip(lower=0)).clip(0,15)
df_oss['jitter_ms_derived'] = 0.18 * df_oss['latency_ms_derived']
df_oss['cell_load_pct_real'] = np.where(
    (df_oss['rat_type']=='4G') & df_oss['active_users'].notna() & df_oss['active_users_max'].notna() & (df_oss['active_users_max']>0),
    (df_oss['active_users'] / df_oss['active_users_max'] * 100).clip(0,100), np.nan)
display(df_oss[['latency_ms_derived','packet_loss_pct_derived','jitter_ms_derived','cell_load_pct_real']].describe().T)

## 11 · OSS aggregates per (area, month, RAT)

In [ ]:
agg = df_oss.groupby(['area','month_year','rat_type']).agg(
    avg_integrity=('integrity','mean'),
    avg_cdr=('call_drop_rate','mean'),
    avg_throughput_mbps=('throughput_mbps','mean'),
    avg_users=('active_users','mean'),
    avg_latency_derived=('latency_ms_derived','mean'),
    avg_loss_derived=('packet_loss_pct_derived','mean'),
    cell_count=('integrity','count'),
    anomaly_count=('anomaly_flag','sum'),
).reset_index()
print(f'Aggregates: {len(agg):,} (area × month × RAT)')
display(agg.head())

## 12 · Write processed/ to MinIO

In [ ]:
def put_pq(df, bucket, key):
    buf = io.BytesIO(); df.to_parquet(buf, index=False, engine='pyarrow')
    buf.seek(0); s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f'  ↑ {bucket}/{key}  ({len(df):,} rows)')
put_pq(df_bss, PROC, 'bss_clean.parquet')
put_pq(df_oss, PROC, 'oss_clean.parquet')
put_pq(agg, PROC, 'oss_aggregates.parquet')

## 13 · Curated DATA WAREHOUSE — `warehouse.parquet`

**L4 Agent's primary read source.** One row per (subscriber, month) joined with area-aggregated OSS KPIs
and engineered targets. Single source of truth.

In [ ]:
oss_am = agg.groupby(['area','month_year']).agg(
    avg_integrity_area=('avg_integrity','mean'),
    avg_cdr_area=('avg_cdr','mean'),
    avg_throughput_area=('avg_throughput_mbps','mean'),
    avg_users_area=('avg_users','mean'),
    avg_latency_area=('avg_latency_derived','mean'),
    avg_loss_area=('avg_loss_derived','mean'),
    cell_count_area=('cell_count','sum'),
    anomaly_count_area=('anomaly_count','sum'),
).reset_index()
warehouse = df_bss.merge(oss_am, on=['area','month_year'], how='left')
warehouse['cem_score_target'] = (
    0.4 * warehouse['attach_gap'].fillna(0).rsub(1).clip(0,1)
  + 0.3 * warehouse['traffic_share_4g'].fillna(0).clip(0,1)
  + 0.2 * warehouse['avg_integrity_area'].fillna(95).div(100).clip(0,1)
  + 0.1 * (1 - warehouse['avg_cdr_area'].fillna(1).clip(0,5).div(5))
).clip(0,1)
warehouse['rat_gap_score'] = (warehouse['is_4g_capable'] * (1.0 - warehouse.get('traffic_share_4g',0).fillna(0))).clip(0,1)
warehouse['churn_risk_flag'] = ((warehouse['cem_score_target']<0.4) & (warehouse['rat_gap_score']>0.5)).astype(int)
put_pq(warehouse, CUR, 'warehouse.parquet')
put_pq(warehouse, CUR, 'subscribers.parquet')  # backward-compat alias
print(f'\nwarehouse: {len(warehouse):,} rows × {warehouse.shape[1]} cols')
print(f'  CEM mean={warehouse.cem_score_target.mean():.3f}  RAT gap >0.5={(warehouse.rat_gap_score>0.5).mean()*100:.1f}%  Churn={warehouse.churn_risk_flag.mean()*100:.2f}%')

## 14 · Curated cells.parquet (per-cell for VAE)

In [ ]:
put_pq(df_oss, CUR, 'cells.parquet')
print(f'cells: {len(df_oss):,} rows × {df_oss.shape[1]} cols')

## 15 · Train/val/test splits — dual policy

**Primary**: random 70/15/15 stratified by (month, RAT-capability). Reports best metrics.
**Production hold-out**: last 10% of months. Reports realistic future-data metrics.

In [ ]:
warehouse = warehouse.reset_index(drop=True)
warehouse['_strata'] = warehouse['month_year'].astype(str) + '_' + warehouse.get('highest_rat','NA').astype(str)
idx = warehouse.index.values
tv, te = train_test_split(idx, test_size=0.15, stratify=warehouse.loc[idx,'_strata'], random_state=42)
tr, va = train_test_split(tv,  test_size=0.176, stratify=warehouse.loc[tv,'_strata'], random_state=42)
sorted_m = sorted(warehouse.month_year.unique())
cutoff = int(len(sorted_m)*0.9)
ho_months = sorted_m[cutoff:]
ho_idx = warehouse.index[warehouse.month_year.isin(ho_months)].tolist()
splits = {
    'random': {'train':tr.tolist(),'val':va.tolist(),'test':te.tolist()},
    'temporal': {'holdout_months':ho_months,'holdout_indices':ho_idx},
    'meta': {'total_rows':int(len(warehouse)),
             'random_sizes':{'train':len(tr),'val':len(va),'test':len(te)},
             'temporal_holdout_size':len(ho_idx)},
}
s3.put_object(Bucket=CUR, Key='splits.json', Body=json.dumps(splits, indent=2).encode())
print('Splits → curated/splits.json')
print(json.dumps(splits['meta'], indent=2))

## 16 · Validation report

In [ ]:
def pq_from_s3(key, bucket=CUR):
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=bucket, Key=key)['Body'].read()))
wh = pq_from_s3('warehouse.parquet')
cc = pq_from_s3('cells.parquet')
rep = {
    'warehouse': {'rows':int(len(wh)),'cols':int(wh.shape[1]),
                  'nan_cols': wh.isna().sum()[wh.isna().sum()>0].to_dict()},
    'cells': {'rows':int(len(cc)),'cols':int(cc.shape[1])},
    'splits_total': sum(len(splits['random'][k]) for k in ['train','val','test']),
    'expected_total': len(warehouse),
}
assert rep['splits_total']==rep['expected_total']
print(json.dumps(rep, indent=2, default=str))
print('\nVALIDATION PASSED.')

---
## Pipeline complete

**MinIO layout**: raw/{bss/+oss/} · processed/{bss+oss+agg}.parquet · curated/{warehouse+cells}.parquet + splits.json

**Next**: open `02_cem_score_training.ipynb` in Jupyter.